# Notebook 07 — 2026 Attendance Forecast and Matchday Demand Planning  
  
Apply the frozen Gradient Boosting pipeline trained through 2025 to:  
  
- forecast attendance for the nine confirmed 2026 AFL Round 24 fixtures in the 17 August 2026 snapshot  
- translate the predictions into relative matchday demand levels.

## 1. Objective and Decision Framework

### 1.1 Business Objective

This notebook applies the frozen final scoring pipeline produced in Notebook 06 to the nine confirmed AFL Round 24 fixtures available in the 17 August 2026 snapshot.

The objective is to:

- generate an attendance forecast for each confirmed fixture;
- translate each forecast into a relative Low, Medium, or High attendance level using the historical AFL attendance distribution; and
- produce a concise fixture-level summary that can support matchday demand review and planning prioritisation.

The predictions are decision-support estimates rather than exact attendance commitments or official operational rules.

### 1.2 Scoring Scope and Guardrails

The following scoring rules are fixed before generating the 2026 forecasts:

- The Gradient Boosting scoring pipeline trained on all available data through 2025 will be used without further tuning or model comparison.
- The 2026 scoring data must contain the same 17 input features, in the same order, as the model input defined in Notebook 06.
- The 2026 fixtures contain no observed attendance target and are used only for forward scoring.
- Prediction checks will assess completeness and practical reasonableness without changing the model.
- MAE, RMSE, and other performance metrics cannot be calculated until actual 2026 attendance becomes available.

### 1.3 Relative Attendance-Level Rules

The attendance-level boundaries are fixed using the distribution of all 2,297 eligible historical AFL matches available through 2025.

The lower and upper tertile boundaries are rounded to the nearest 1,000 attendees, producing the following rules:

| Attendance level | Predicted attendance |
|---|---:|
| Low | Below 27,000 |
| Medium | 27,000 to below 40,000 |
| High | 40,000 or above |

These levels are relative review signals for the nine AFL fixtures. They are not official club operating thresholds, venue-capacity classifications, or direct staffing rules.

## 2. Load and Validate Scoring Inputs

### 2.1 Setup and File Paths

This notebook uses the frozen scoring model produced in Notebook 06 and the processed 2026 fixture dataset. File paths are defined centrally to keep the scoring workflow reproducible.

In [1]:
# Import the libraries required for path handling, model loading, and fixture-level scoring.
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

# Configure pandas for readable notebook table outputs.
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

In [2]:
# Resolve the repository root when the notebook is launched from either supported location.
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Define the frozen model and processed datasets required for scoring.
MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "final"
    / "notebook_06_scoring_model.joblib"
)

SCORING_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "squiggle_2026_scoring_dataset.csv"
)

HISTORICAL_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "historical_model_dataset.csv"
)

# Confirm that all required local artifacts are available.
print(f"Project root: {PROJECT_ROOT}")
print(f"Model exists: {MODEL_PATH.exists()}")
print(f"2026 scoring data exists: {SCORING_DATA_PATH.exists()}")
print(f"Historical data exists: {HISTORICAL_DATA_PATH.exists()}")

Project root: C:\Users\dajun\Projects\AFL_Project
Model exists: True
2026 scoring data exists: True
Historical data exists: True


### 2.2 Load the Frozen Scoring Model

The frozen scoring bundle was refitted on all available data through 2025 after model selection and locked-test evaluation were completed in Notebook 06.

It is loaded here only for forward scoring. Its model configuration will not be modified.

In [3]:
# Load the frozen scoring bundle exported by Notebook 06.
scoring_bundle = joblib.load(MODEL_PATH)

# Confirm the artifact type and available metadata fields before scoring.
print("Scoring bundle loaded successfully.")
print(f"Bundle type: {type(scoring_bundle).__name__}")
print(f"Bundle keys: {list(scoring_bundle.keys())}")

Scoring bundle loaded successfully.
Bundle type: dict
Bundle keys: ['estimator', 'model_role', 'model_key', 'model_name', 'selected_parameters', 'feature_columns', 'numerical_features', 'categorical_features', 'target_column', 'selection_metric', 'training_rows', 'training_seasons', 'trained_through', 'evaluation_summary']


Extract the fitted estimator and its input-feature metadata from the saved bundle. The training details are displayed to confirm that the correct final scoring artifact has been loaded.

In [4]:
# Extract the frozen estimator and its model-input contract.
scoring_estimator = scoring_bundle["estimator"]
feature_columns = list(scoring_bundle["feature_columns"])
numerical_features = list(scoring_bundle["numerical_features"])
categorical_features = list(scoring_bundle["categorical_features"])
target_column = scoring_bundle["target_column"]

# Display key metadata to verify the provenance and scope of the scoring artifact.
print(f"Model name: {scoring_bundle['model_name']}")
print(f"Model role: {scoring_bundle['model_role']}")
print(f"Estimator type: {type(scoring_estimator).__name__}")
print(f"Trained through: {scoring_bundle['trained_through']}")
print(f"Training rows: {scoring_bundle['training_rows']:,}")
print(f"Training seasons: {scoring_bundle['training_seasons']}")
print(f"Target column: {target_column}")
print(f"Feature count: {len(feature_columns)}")
print(f"Numerical features: {len(numerical_features)}")
print(f"Categorical features: {len(categorical_features)}")

Model name: Gradient Boosting Regressor
Model role: 2026_scoring_model
Estimator type: Pipeline
Trained through: 2025
Training rows: 2,297
Training seasons: [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2022, 2023, 2024, 2025]
Target column: attendance
Feature count: 17
Numerical features: 9
Categorical features: 8


### 2.3 Load and Validate the 2026 Scoring Dataset

Load the processed 2026 fixture dataset and verify that it is compatible with the frozen scoring pipeline. The checks are limited to the feature schema, input completeness, target exclusion, identifier uniqueness, and season scope required for reliable forward scoring.

In [5]:
# Load the processed fixture-level dataset for 2026 forward scoring.
scoring_2026 = pd.read_csv(SCORING_DATA_PATH)

# Identify any required model features that are unavailable in the scoring data.
missing_features = [
    column
    for column in feature_columns
    if column not in scoring_2026.columns
]

# Validate the essential schema and scope conditions before prediction.
assert not missing_features, (
    f"Required model features are missing: {missing_features}"
)
assert target_column not in scoring_2026.columns, (
    f"The target column {target_column!r} must not be present in future scoring data."
)
assert scoring_2026["match_id"].is_unique, (
    "The scoring dataset contains duplicate match_id values."
)
assert scoring_2026["season_year"].eq(2026).all(), (
    "The scoring dataset contains records outside the 2026 season."
)

# Construct the model-input matrix using the saved feature order.
X_2026 = scoring_2026.loc[:, feature_columns].copy()

# Confirm that all values required by the scoring pipeline are complete.
assert not X_2026.isna().any().any(), (
    "The model-input matrix contains missing values."
)

# Report the validated scoring-data dimensions.
print("2026 scoring data loaded and validated successfully.")
print(f"Fixture rows: {len(scoring_2026):,}")
print(f"Dataset columns: {scoring_2026.shape[1]:,}")
print(f"Model-input features: {X_2026.shape[1]:,}")

display(scoring_2026.head())

2026 scoring data loaded and validated successfully.
Fixture rows: 9
Dataset columns: 22
Model-input features: 17


,snapshot_date,source_game_id,match_id,match_date,start_time,season_year,start_hour,home_team_last_5_attendance_mean,away_team_last_5_attendance_mean,venue_last_10_attendance_mean,home_team_last_5_win_rate,away_team_last_5_win_rate,home_team_last_5_score_margin_mean,away_team_last_5_score_margin_mean,round_label,home_team,away_team,venue_name,match_month,match_day_of_week,is_night_match,is_school_holiday
0,2026-08-17,38697,20260820_H15_A08,2026-08-20,19:30:00,2026,19.500000,27522.6,33561.4,25785.7,0.4,0.2,-2.0,-14.2,Round 24,St Kilda,Gold Coast,Docklands,8,Thursday,True,False
1,2026-08-17,38692,20260821_H04_A02,2026-08-21,19:40:00,2026,19.666667,66323.8,70224.6,65984.5,0.6,0.8,9.8,23.2,Round 24,Collingwood,Brisbane Lions,MCG,8,Friday,True,False
2,2026-08-17,38693,20260822_H03_A06,2026-08-22,13:15:00,2026,13.250000,36183.8,45125.2,25785.7,0.6,0.8,22.4,31.0,Round 24,Carlton,Fremantle,Docklands,8,Saturday,False,False
3,2026-08-17,38696,20260822_H11_A18,2026-08-22,16:15:00,2026,16.250000,37635.8,30654.8,65984.5,0.8,0.4,17.0,1.2,Round 24,Melbourne,Western Bulldogs,MCG,8,Saturday,False,False
4,2026-08-17,38699,20260822_H01_A09,2026-08-22,19:40:00,2026,19.666667,41762.8,16571.0,44454.2,0.6,0.4,7.8,2.0,Round 24,Adelaide,Greater Western Sydney,Adelaide Oval,8,Saturday,True,False


### 2.4 Establish Historical Attendance Cutoffs

Use the observed attendance distribution from the same 2,297 eligible historical AFL matches used to fit the final 2026 scoring model. The lower and upper tertiles are calculated and rounded to the nearest 1,000 attendees before any 2026 predictions are generated.

In [6]:
# Load the historical dataset used to fit the final 2026 scoring model.
historical_model_data = pd.read_csv(HISTORICAL_DATA_PATH)

# Confirm that the reference population matches the final model metadata.
assert len(historical_model_data) == scoring_bundle["training_rows"], (
    "The historical reference row count does not match the scoring model."
)
assert target_column in historical_model_data.columns, (
    f"The historical dataset does not contain {target_column!r}."
)

# Extract and validate the observed historical attendance values.
historical_attendance = pd.to_numeric(
    historical_model_data[target_column],
    errors="coerce",
)

assert historical_attendance.notna().all(), (
    "The historical attendance reference contains missing or invalid values."
)
assert historical_attendance.gt(0).all(), (
    "The historical attendance reference contains non-positive values."
)

# Calculate the lower and upper tertiles of historical AFL attendance.
lower_cutoff_raw, upper_cutoff_raw = (
    historical_attendance
    .quantile([1 / 3, 2 / 3])
    .tolist()
)

# Round the data-derived boundaries to the nearest 1,000 attendees.
lower_cutoff = int(round(lower_cutoff_raw, -3))
upper_cutoff = int(round(upper_cutoff_raw, -3))

assert lower_cutoff < upper_cutoff, (
    "The calculated attendance-level boundaries are invalid."
)

# Apply the fixed boundaries to confirm the historical tier distribution.
historical_levels = pd.cut(
    historical_attendance,
    bins=[-np.inf, lower_cutoff, upper_cutoff, np.inf],
    labels=["Low", "Medium", "High"],
    right=False,
)

historical_level_summary = (
    historical_levels
    .value_counts(sort=False)
    .rename_axis("attendance_level")
    .reset_index(name="historical_matches")
)

historical_level_summary["percentage"] = (
    historical_level_summary["historical_matches"]
    .div(len(historical_attendance))
    .mul(100)
    .round(1)
)

# Report the raw percentiles, fixed cutoffs, and resulting historical distribution.
print(f"Historical reference rows: {len(historical_attendance):,}")
print(f"Raw lower tertile: {lower_cutoff_raw:,.2f}")
print(f"Raw upper tertile: {upper_cutoff_raw:,.2f}")
print(f"Fixed lower cutoff: {lower_cutoff:,}")
print(f"Fixed upper cutoff: {upper_cutoff:,}")

display(historical_level_summary)

Historical reference rows: 2,297
Raw lower tertile: 26,722.67
Raw upper tertile: 39,964.67
Fixed lower cutoff: 27,000
Fixed upper cutoff: 40,000


,attendance_level,historical_matches,percentage
0,Low,777,33.8
1,Medium,756,32.9
2,High,764,33.3


### 2.5 Prepare the Attendance-Level Reference Table

Convert the fixed cutoffs and historical distribution established above into a reporting-ready reference table for downstream Power BI use. This step packages the approved demand-level policy and its historical context without recalculating or changing the thresholds.

In [7]:
# Define the ordered demand-level metadata using the fixed historical cutoffs.
level_metadata = pd.DataFrame(
    {
        "attendance_level": ["Low", "Medium", "High"],
        "lower_bound": pd.array(
            [
                pd.NA,
                lower_cutoff,
                upper_cutoff,
            ],
            dtype="Int64",
        ),
        "upper_bound": pd.array(
            [
                lower_cutoff - 1,
                upper_cutoff - 1,
                pd.NA,
            ],
            dtype="Int64",
        ),
        "sort_order": [1, 2, 3],
    }
)

# Prepare the historical match counts calculated in Section 2.4.
historical_level_counts = (
    historical_level_summary[
        [
            "attendance_level",
            "historical_matches",
        ]
    ]
    .copy()
)

historical_level_counts["attendance_level"] = (
    historical_level_counts["attendance_level"]
    .astype("string")
)

# Combine the fixed boundaries with their historical distribution.
attendance_level_reference = (
    level_metadata
    .merge(
        historical_level_counts,
        on="attendance_level",
        how="left",
        validate="one_to_one",
    )
)

# Store the historical percentage as a decimal ratio for Power BI formatting.
attendance_level_reference["historical_percentage"] = (
    attendance_level_reference["historical_matches"]
    .div(len(historical_attendance))
)

# Confirm that the reporting table preserves the complete reference population.
assert attendance_level_reference[
    "historical_matches"
].sum() == len(historical_attendance), (
    "The attendance-level counts do not match the historical population."
)

assert np.isclose(
    attendance_level_reference[
        "historical_percentage"
    ].sum(),
    1.0,
), (
    "The attendance-level percentages do not sum to 100%."
)

# Present a reader-friendly copy while preserving decimal ratios for export.
attendance_level_reference_display = (
    attendance_level_reference.copy()
)

attendance_level_reference_display[
    "historical_percentage"
] = (
    attendance_level_reference_display[
        "historical_percentage"
    ]
    .mul(100)
    .round(1)
)

display(attendance_level_reference_display)

,attendance_level,lower_bound,upper_bound,sort_order,historical_matches,historical_percentage
0,Low,<NA>,26999,1,777,33.8
1,Medium,27000,39999,2,756,32.9
2,High,40000,<NA>,3,764,33.3


## 3. Generate 2026 Attendance Forecasts

### 3.1 Score the Confirmed Fixtures

Generate one attendance estimate for each of the nine confirmed fixtures using the frozen scoring pipeline. The pipeline applies the preprocessing and Gradient Boosting model fitted on all eligible historical data through 2025. No additional fitting, tuning, or model selection is performed.

In [8]:
# Generate one attendance prediction for each confirmed 2026 fixture.
attendance_predictions = np.asarray(
    scoring_estimator.predict(X_2026),
    dtype="float64",
)

# Confirm that the frozen pipeline returned one finite prediction per fixture.
assert len(attendance_predictions) == len(scoring_2026), (
    "The number of predictions does not match the number of scoring fixtures."
)
assert np.isfinite(attendance_predictions).all(), (
    "The scoring pipeline produced one or more invalid predictions."
)

# Report completion without modifying or evaluating the frozen model.
print("2026 attendance predictions generated successfully.")
print(f"Fixtures scored: {len(attendance_predictions):,}")

2026 attendance predictions generated successfully.
Fixtures scored: 9


### 3.2 Build the Forecast Results Table

Combine the model predictions with the fixture metadata required for interpretation and downstream reporting. Predicted attendance is rounded to the nearest attendee for presentation, while the original model outputs remain available in `attendance_predictions`.

In [9]:
# Retain the fixture metadata required for traceability and reporting.
forecast_results = scoring_2026[
    [
        "snapshot_date",
        "match_id",
        "match_date",
        "start_time",
        "round_label",
        "home_team",
        "away_team",
        "venue_name",
    ]
].copy()

# Add the model predictions as whole-attendee planning estimates.
forecast_results["predicted_attendance"] = np.rint(
    attendance_predictions
).astype("int64")

# Standardise the date fields and preserve chronological fixture order.
forecast_results["snapshot_date"] = pd.to_datetime(
    forecast_results["snapshot_date"],
    errors="raise",
)

forecast_results["match_date"] = pd.to_datetime(
    forecast_results["match_date"],
    errors="raise",
)

forecast_results = (
    forecast_results
    .sort_values(
        ["match_date", "start_time", "match_id"],
        kind="stable",
    )
    .reset_index(drop=True)
)

# Present a concise fixture-level forecast table.
forecast_preview_columns = [
    "match_date",
    "start_time",
    "home_team",
    "away_team",
    "venue_name",
    "predicted_attendance",
]

display(forecast_results[forecast_preview_columns])

,match_date,start_time,home_team,away_team,venue_name,predicted_attendance
0,2026-08-20,19:30:00,St Kilda,Gold Coast,Docklands,21675
1,2026-08-21,19:40:00,Collingwood,Brisbane Lions,MCG,77848
2,2026-08-22,13:15:00,Carlton,Fremantle,Docklands,34774
3,2026-08-22,16:15:00,Melbourne,Western Bulldogs,MCG,39611
4,2026-08-22,19:40:00,Adelaide,Greater Western Sydney,Adelaide Oval,46296
5,2026-08-22,19:45:00,Geelong,Richmond,Kardinia Park,36723
6,2026-08-23,12:20:00,Essendon,Port Adelaide,Docklands,27082
7,2026-08-23,15:20:00,Sydney,North Melbourne,SCG,34153
8,2026-08-23,17:20:00,West Coast,Hawthorn,Perth Stadium,44182


## 4. Review Forecast Reasonableness

### 4.1 Forecast Range Check

Check that all forecasts are positive and compare the 2026 forecast range with the observed historical attendance range. The historical range is used only as a plausibility reference and does not constrain the model outputs.

In [10]:
# Confirm that every reported attendance forecast is positive.
assert forecast_results["predicted_attendance"].gt(0).all(), (
    "One or more predicted attendance values are non-positive."
)

# Compare the forecast range with the observed historical attendance range.
historical_min = int(historical_attendance.min())
historical_max = int(historical_attendance.max())
forecast_min = int(forecast_results["predicted_attendance"].min())
forecast_max = int(forecast_results["predicted_attendance"].max())

within_historical_range = (
    historical_min <= forecast_min
    and forecast_max <= historical_max
)

# Report the comparison without modifying any model predictions.
print(f"Historical attendance range: {historical_min:,} to {historical_max:,}")
print(f"2026 forecast range: {forecast_min:,} to {forecast_max:,}")
print(f"Forecast range within historical range: {within_historical_range}")

Historical attendance range: 3,413 to 100,024
2026 forecast range: 21,675 to 77,848
Forecast range within historical range: True


### 4.2 Fixture-Level Context Review

Review each forecast alongside the home team's last-five, away team's last-five, and venue's last-ten attendance means. These benchmarks support a fixture-level plausibility review; they are not actual 2026 outcomes or causal explanations of the predictions.

In [11]:
# Join the recent attendance benchmarks already used as model inputs.
context_columns = [
    "match_id",
    "home_team_last_5_attendance_mean",
    "away_team_last_5_attendance_mean",
    "venue_last_10_attendance_mean",
]

fixture_context_review = forecast_results.merge(
    scoring_2026[context_columns],
    on="match_id",
    how="left",
    validate="one_to_one",
)

# Measure each forecast against the venue's recent attendance benchmark.
fixture_context_review["difference_from_venue_mean"] = (
    fixture_context_review["predicted_attendance"]
    - fixture_context_review["venue_last_10_attendance_mean"]
).round().astype("int64")

# Round the contextual attendance averages for presentation.
attendance_benchmark_columns = [
    "home_team_last_5_attendance_mean",
    "away_team_last_5_attendance_mean",
    "venue_last_10_attendance_mean",
]

fixture_context_review[attendance_benchmark_columns] = (
    fixture_context_review[attendance_benchmark_columns]
    .round()
    .astype("int64")
)

# Rank fixtures by predicted attendance for a concise reasonableness review.
context_review_columns = [
    "match_date",
    "home_team",
    "away_team",
    "venue_name",
    "home_team_last_5_attendance_mean",
    "away_team_last_5_attendance_mean",
    "venue_last_10_attendance_mean",
    "predicted_attendance",
    "difference_from_venue_mean",
]

display(
    fixture_context_review[
        context_review_columns
    ].sort_values(
        "predicted_attendance",
        ascending=False,
    )
)

,match_date,home_team,away_team,venue_name,home_team_last_5_attendance_mean,away_team_last_5_attendance_mean,venue_last_10_attendance_mean,predicted_attendance,difference_from_venue_mean
1,2026-08-21,Collingwood,Brisbane Lions,MCG,66324,70225,65984,77848,11864
4,2026-08-22,Adelaide,Greater Western Sydney,Adelaide Oval,41763,16571,44454,46296,1842
8,2026-08-23,West Coast,Hawthorn,Perth Stadium,32172,51633,44789,44182,-607
3,2026-08-22,Melbourne,Western Bulldogs,MCG,37636,30655,65984,39611,-26374
5,2026-08-22,Geelong,Richmond,Kardinia Park,73944,34595,30108,36723,6615
2,2026-08-22,Carlton,Fremantle,Docklands,36184,45125,25786,34774,8988
7,2026-08-23,Sydney,North Melbourne,SCG,28890,19569,33674,34153,479
6,2026-08-23,Essendon,Port Adelaide,Docklands,28024,34078,25786,27082,1296
0,2026-08-20,St Kilda,Gold Coast,Docklands,27523,33561,25786,21675,-4111


## 5. Translate Forecasts into Matchday Demand Levels

### 5.1 Apply the Predefined Thresholds

Apply the fixed historical attendance cutoffs established in Section 2.4 to classify each fixture as Low, Medium, or High. The levels provide a consistent relative prioritisation signal across the nine fixtures.

In [12]:
# Apply the fixed pre-scoring thresholds to each attendance forecast.
forecast_results["attendance_level"] = pd.cut(
    forecast_results["predicted_attendance"],
    bins=[
        -np.inf,
        lower_cutoff,
        upper_cutoff,
        np.inf,
    ],
    labels=[
        "Low",
        "Medium",
        "High",
    ],
    right=False,
)

# Confirm that every fixture received an attendance-level classification.
assert forecast_results["attendance_level"].notna().all(), (
    "One or more fixtures could not be assigned an attendance level."
)

# Present the classified fixtures from highest to lowest forecast demand.
classified_forecast_columns = [
    "match_date",
    "start_time",
    "home_team",
    "away_team",
    "venue_name",
    "predicted_attendance",
    "attendance_level",
]

display(
    forecast_results[
        classified_forecast_columns
    ].sort_values(
        "predicted_attendance",
        ascending=False,
    )
)

,match_date,start_time,home_team,away_team,venue_name,predicted_attendance,attendance_level
1,2026-08-21,19:40:00,Collingwood,Brisbane Lions,MCG,77848,High
4,2026-08-22,19:40:00,Adelaide,Greater Western Sydney,Adelaide Oval,46296,High
8,2026-08-23,17:20:00,West Coast,Hawthorn,Perth Stadium,44182,High
3,2026-08-22,16:15:00,Melbourne,Western Bulldogs,MCG,39611,Medium
5,2026-08-22,19:45:00,Geelong,Richmond,Kardinia Park,36723,Medium
2,2026-08-22,13:15:00,Carlton,Fremantle,Docklands,34774,Medium
7,2026-08-23,15:20:00,Sydney,North Melbourne,SCG,34153,Medium
6,2026-08-23,12:20:00,Essendon,Port Adelaide,Docklands,27082,Medium
0,2026-08-20,19:30:00,St Kilda,Gold Coast,Docklands,21675,Low


### 5.2 Summarise Matchday Planning Priorities

Summarise the number of fixtures assigned to each attendance level and their average predicted attendance. This provides a portfolio-level view of relative demand while leaving detailed staffing and venue decisions to operational stakeholders.

In [13]:
# Summarise the fixture count and average forecast within each attendance level.
planning_priority_summary = (
    forecast_results
    .groupby(
        "attendance_level",
        observed=True,
    )
    .agg(
        fixture_count=("match_id", "count"),
        average_predicted_attendance=(
            "predicted_attendance",
            "mean",
        ),
    )
    .reset_index()
)

# Present the average forecasts as whole-attendee planning estimates.
planning_priority_summary["average_predicted_attendance"] = (
    planning_priority_summary["average_predicted_attendance"]
    .round()
    .astype("int64")
)

# Order the summary from the highest to the lowest forecast demand.
planning_priority_summary = (
    planning_priority_summary
    .sort_values(
        "average_predicted_attendance",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(planning_priority_summary)

,attendance_level,fixture_count,average_predicted_attendance
0,High,3,56109
1,Medium,5,34469
2,Low,1,21675


## 6. Export and Conclude

### 6.1 Export the Matchday Planning Summary

Export the chronologically ordered fixture forecasts and the fixed attendance-level reference table as validated machine-readable outputs. A reporting copy of the forecasts is retained alongside the reference table to provide Power BI with a self-contained and reproducible input layer.

In [ ]:
# Define the canonical forecast and Power BI reporting locations.
FORECAST_OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "predictions"
    / "afl_2026_round_24_attendance_forecasts.csv"
)

POWER_BI_DATA_DIR = (
    PROJECT_ROOT
    / "reports"
    / "powerbi"
    / "data"
)

POWER_BI_FORECAST_PATH = (
    POWER_BI_DATA_DIR
    / "forecasts.csv"
)

ATTENDANCE_LEVEL_REFERENCE_OUTPUT_PATH = (
    POWER_BI_DATA_DIR
    / "attendance_level_reference.csv"
)

# Create the required output directories.
FORECAST_OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

POWER_BI_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# Retain only the fields required for the final planning handoff.
planning_export_columns = [
    "snapshot_date",
    "match_id",
    "match_date",
    "start_time",
    "round_label",
    "home_team",
    "away_team",
    "venue_name",
    "predicted_attendance",
    "attendance_level",
]

planning_export = (
    forecast_results[planning_export_columns]
    .sort_values(
        [
            "match_date",
            "start_time",
            "match_id",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

# Confirm that the reporting outputs preserve their required contracts.
assert len(planning_export) == len(scoring_2026), (
    "The forecast export does not contain all scoring fixtures."
)

assert planning_export["match_id"].is_unique, (
    "The forecast export contains duplicate match identifiers."
)

assert attendance_level_reference[
    "attendance_level"
].tolist() == [
    "Low",
    "Medium",
    "High",
], (
    "The attendance-level reference order is invalid."
)

# Export the canonical fixture-level forecast summary.
planning_export.to_csv(
    FORECAST_OUTPUT_PATH,
    index=False,
    date_format="%Y-%m-%d",
)

# Export the self-contained Power BI forecast input.
planning_export.to_csv(
    POWER_BI_FORECAST_PATH,
    index=False,
    date_format="%Y-%m-%d",
)

# Export the fixed demand-level policy and its historical context.
attendance_level_reference.to_csv(
    ATTENDANCE_LEVEL_REFERENCE_OUTPUT_PATH,
    index=False,
)

# Present a concise record of the exported machine-readable outputs.
exported_output_summary = pd.DataFrame(
    [
        {
            "artifact": "Canonical forecast summary",
            "path": str(
                FORECAST_OUTPUT_PATH.relative_to(
                    PROJECT_ROOT
                )
            ),
            "rows": len(planning_export),
        },
        {
            "artifact": "Power BI forecast input",
            "path": str(
                POWER_BI_FORECAST_PATH.relative_to(
                    PROJECT_ROOT
                )
            ),
            "rows": len(planning_export),
        },
        {
            "artifact": "Attendance-level reference",
            "path": str(
                ATTENDANCE_LEVEL_REFERENCE_OUTPUT_PATH.relative_to(
                    PROJECT_ROOT
                )
            ),
            "rows": len(attendance_level_reference),
        },
    ]
)

display(exported_output_summary)

,artifact,path,rows
0,Canonical forecast summary,data\processed\predictions\afl_2026_round_24_a...,9
1,Power BI forecast input,reports\powerbi\data\forecasts.csv,9
2,Attendance-level reference,reports\powerbi\data\attendance_level_referenc...,3


### 6.2 Final Interpretation

Across the nine confirmed fixtures, the model assigns three matches to High, five to Medium, and one to Low relative attendance demand.

Collingwood vs Brisbane at the MCG has the highest forecast at 77,848 attendees. Adelaide vs Greater Western Sydney (46,296) and West Coast vs Hawthorn (44,182) are the other High-level fixtures and therefore represent the strongest relative demand signals in the snapshot.

Sydney vs North Melbourne at the SCG is forecast at 34,153 attendees and is classified as Medium. St Kilda vs Gold Coast at Docklands has the lowest forecast at 21,675 attendees and is the only Low-level fixture.

The ranking can help sequence further matchday review, but the forecasts should be combined with current ticketing and operational information before final decisions are made.

### 6.3 Limitations and Handoff

The forecasts should be interpreted within the following boundaries:

- They reflect the frozen 17 August 2026 data snapshot and do not capture later ticketing, fixture, or operational changes.
- Actual 2026 attendance is not available at scoring time, so forecast accuracy cannot yet be evaluated. The locked 2025 test MAE of approximately 4,561 attendees provides historical performance context, not a fixture-specific error bound.
- The model does not include live ticket sales, membership reservations, late weather conditions, transport disruptions, or other event-specific operational signals.
- The Low, Medium, and High labels are relative historical attendance levels rather than official venue-capacity or staffing rules.

The exported file provides the scoring handoff for the nine confirmed fixtures. Once actual attendance becomes available, it should be joined by `match_id` to measure forecast error, review the attendance-level assignments, and determine whether the model or thresholds require updating.